# 09-bcr-influenza-data-loading

In [1]:
import scanpy as sc
import scirpy as ir
import numpy as np
import json
from pathlib import Path
import pandas as pd
import muon as mu
import anndata as ad
import tarfile
import re
import warnings
warnings.filterwarnings('ignore')

DATA = Path("data")

/Users/alegator1209/micromamba/envs/pytcr/lib/python3.12/site-packages/h5py/__init__.py:36: UserWarning: h5py is running against HDF5 2.2.0 when it was built against 2.1.0, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "
Matplotlib is building the font cache; this may take a moment.
/Users/alegator1209/micromamba/envs/pytcr/lib/python3.12/site-packages/muon/_core/preproc.py:32: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):


In [2]:
metadata = pd.read_csv(DATA / "metadata.csv", dtype={"patient": str}).set_index("patient")
metadata

,sex,age,ethnicity,race
patient,,,,
09,female,25,Not Hispanic or Latino,white
15,male,29,Hispanic or Latino,white
48,male,85,Not Hispanic or Latino,white
67,female,28,Not Hispanic or Latino,white
93,female,92,Not Hispanic or Latino,white
94,female,66,Not Hispanic or Latino,white


In [3]:
def extract_tar(tar_path: Path, dest: Path) -> Path:
  if not dest.exists():
    with tarfile.open(tar_path, "r:gz") as tf:
      tf.extractall(dest)
  return dest

def read_sample(path: Path) -> tuple[ad.AnnData, ad.AnnData]:
  cellranger_tar = next(path.glob("*_cellranger.tar.gz"))
  bcrvdj_tar = next(path.glob("*_BCRVDJ.tar.gz"))

  cellranger_dir = extract_tar(cellranger_tar, path / "cellranger")
  bcrvdj_dir = extract_tar(bcrvdj_tar, path / "BCRVDJ")
  contig_csv = next(bcrvdj_dir.glob("**/filtered_contig_annotations.csv"))

  adata_gex = sc.read_10x_mtx(cellranger_dir, var_names="gene_symbols", compressed=False)
  adata_gex.var_names_make_unique()

  adata_tcr = ir.io.read_10x_vdj(contig_csv)
  ir.pp.index_chains(adata_tcr)
  ir.tl.chain_qc(adata_tcr)

  return adata_gex, adata_tcr

def add_metadata(mdata: mu.MuData):
  cell_samples = mdata.obs_names.to_series().str.split("_", expand=True)[1]
  cell_patient = cell_samples.str.split("-", expand=True)[0]

  mdata.obs["sample"] = cell_samples
  mdata.obs["patient"] = cell_patient
  mdata.obs["days_after_vaccination"] = cell_samples.str.split("-", expand=True)[1].astype(int)
  mdata.obs["age"] = cell_patient.map(metadata["age"])
  mdata.obs["sex"] = cell_patient.map(metadata["sex"])
  mdata.obs["young"] = mdata.obs["age"] < 40
  mdata.push_obs()

In [4]:
adatas_gex = {}
adatas_tcr = {}

for sample_dir in sorted(DATA.iterdir()):
  match = re.match(r"(\d+)-(\d+)", sample_dir.name)

  if not sample_dir.is_dir() or not match:
      continue

  sample = sample_dir.name
  gex, tcr = read_sample(sample_dir)

  adatas_gex[sample] = gex
  adatas_tcr[sample] = tcr

adata_gex = ad.concat(adatas_gex, index_unique="_")
adata_tcr = ad.concat(adatas_tcr, index_unique="_")

mdata = mu.MuData({"gex": adata_gex, "airr": adata_tcr})
add_metadata(mdata)
mdata

MuData object with n_obs × n_vars = 140699 × 33538
  obs:	'sample', 'patient', 'days_after_vaccination', 'age', 'sex', 'young'
  2 modalities
    gex:	123693 × 33538
      obs:	'sample', 'patient', 'days_after_vaccination', 'age', 'sex', 'young'
      layers:	None
    airr:	121533 × 0
      obs:	'receptor_type', 'receptor_subtype', 'chain_pairing', 'age', 'days_after_vaccination', 'patient', 'sample', 'sex', 'young'
      obsm:	'airr', 'chain_indices'

In [5]:
n_cells_rna_seq = mdata['gex'].shape[0]
n_cells_bcr_seq = mdata['airr'].shape[0]

print(f"Number of scRNA-seq cells: {n_cells_rna_seq}")
print(f"Number of BCR-seq cells: {n_cells_bcr_seq}")

Number of scRNA-seq cells: 123693
Number of BCR-seq cells: 121533


In [6]:
obs = mdata['airr'].obs
n_cells_bcr_young = obs[obs['young']].shape[0]
print(f"Number of young patients' cells with BCR-seq data: {n_cells_bcr_young}")

Number of young patients' cells with BCR-seq data: 71655


In [7]:
obs = mdata['gex'].obs
n_cells_rna_old_post_vac = obs[~obs['young'] & (obs['days_after_vaccination'] == 7)].shape[0]
print(f"Number of old post-vaccination patients' cells with RNA-seq data: {n_cells_rna_old_post_vac}")

Number of old post-vaccination patients' cells with RNA-seq data: 24175


In [8]:
output = {
  "n_cells_rna_seq": n_cells_rna_seq,
  "n_cells_bcr_seq": n_cells_bcr_seq,
  "n_cells_bcr_young": n_cells_bcr_young,
  "n_cells_rna_old_post_vac": n_cells_rna_old_post_vac
}

print(json.dumps(output, indent=2))

# with open('output.json', 'w') as f:
#     json.dump(output, f, indent=2)
# print('Results saved to output.json:')

{
  "n_cells_rna_seq": 123693,
  "n_cells_bcr_seq": 121533,
  "n_cells_bcr_young": 71655,
  "n_cells_rna_old_post_vac": 24175
}
